# 미션 1_Python 연습

파이썬 기초 문법부터 클래스 설계까지 **19문제** 풀이와 실행 결과 검증.

<img src="images/m1_map.png" alt="미션 1 학습 지도: 6개 파트 19문제" width="726">

**단계별 구성**: 한글 요약 → 실행 흐름 → 코드(이유 주석) → 실행 결과 → 결과 해설 → 소속별 최종 요약표

> 실행 환경: Python 3.10 이상, 표준 라이브러리만 사용

## 🎯 학습 목표

이 미션을 마치면 다음을 직접 할 수 있어야 함

1. 문제에서 정한 **함수 형식 · 출력 형식**을 그대로 지키는 함수를 작성한다
2. 잘못된 입력을 `try · except`와 조건문으로 걸러, 프로그램이 멈추지 않고 안내하게 만든다
3. 반복문 대신 공식 · 내장 함수 · 표준 라이브러리로 더 짧고 빠른 풀이를 고른다
4. 클래스 변수 · 클래스 메서드 · `__str__` · 사용자 정의 예외로 상태와 오류를 관리한다
5. 다른 방법으로 구한 답과 `assert`로 비교해 **결과가 맞다는 근거**를 남긴다

## 🧭 활용처

| 분야 | 이 미션의 기법이 쓰이는 곳 |
| :-- | :-- |
| **입력 검증** | 회원 가입 · 결제 금액 · 예약 날짜처럼 사용자가 직접 넣는 값 처리 |
| **업무 자동화** | CSV 할 일 목록 정리, 정해진 시간 안의 작업 선택 |
| **서비스 설계** | 계좌 · 예약 · 도서 대여처럼 상태가 바뀌는 기능을 클래스로 구성 |
| **노트북 재현** | `input()`을 자동으로 채워 Restart & Run All 한 번으로 전체 실행 |

---
## 0. 준비

### 0-1. 입력 자동화 도구

**한글 요약**

`input()`이 필요한 문제를 정해 둔 값으로 실행하는 도구. 덕분에 **Restart & Run All**로 노트북 전체를 한 번에 실행.

**실행 흐름**

1. **입력**: 미리 정해 둔 입력값
2. **처리**: 원래 `input()` 보관 → 정해 둔 값을 차례로 돌려주는 가짜 `input()`으로 교체 → with 블록 실행
3. **저장**: `fake_inputs` (with 블록이 끝나면 원래 `input()` 복구)
4. **출력**: 질문 + 입력값, 동작 확인 결과

In [1]:
# 모듈 불러오기
import builtins  # 내장 input()을 잠시 바꿔 끼우기 위해
from contextlib import contextmanager  # with 블록의 시작 · 끝 동작을 함수 하나로 정의하기 위해

# 입력 자동화 함수
@contextmanager
def fake_inputs(*values):  # 입력값 개수와 상관없이 받기 위해 가변 인자
    """with 블록 안의 input()을 values로 차례 대체, 입력값도 화면에 표시"""
    remaining = iter(values)  # 정해 둔 입력값을 하나씩 꺼내기 위해
    real_input = builtins.input  # 블록이 끝나면 되돌리기 위해 원래 input() 보관
    def _fake_input(prompt=""):  # 원래 input()과 같은 모양으로 호출되도록
        value = next(remaining)  # 호출될 때마다 다음 입력값을 주기 위해
        print(f"{prompt}{value}")  # 실제로 입력한 것처럼 질문과 값을 함께 보이기 위해
        return value
    builtins.input = _fake_input  # 블록 안의 input()이 정해 둔 값을 돌려주도록
    try:  # 블록에서 오류가 나도 복구 단계가 실행되도록
        yield  # with 블록 안의 코드를 실행하기 위해
    finally:  # 성공 · 실패와 상관없이 마지막에 실행하기 위해
        builtins.input = real_input  # 오류가 나도 원래 input()으로 복구되도록

# 동작 확인
with fake_inputs("hello"):
    assert input("입력: ") == "hello"  # 정해 둔 값이 반환되는지 확인하기 위해
print("✅ 통과")  # 검증을 통과했음을 보이기 위해

입력: hello
✅ 통과


**결과 해설**

```
입력: hello : 가짜 input()이 질문과 정해 둔 값을 함께 출력
✅ 통과     : input()이 "hello"를 반환, with 블록 뒤 원래 input() 복구
```

**소속별 최종 요약표**

| 함수·문법 | 소속 | 역할 | 코드에서 사용한 위치 |
| :-- | :-- | :-- | :-- |
| fake_inputs() | 사용자 정의 함수 | 입력값 자동 공급 | `with fake_inputs("hello"):` |
| @contextmanager | contextlib | with 블록용 함수로 변환 | `@contextmanager` |
| builtins.input | builtins | 내장 input() 교체 · 복구 | `builtins.input = _fake_input` |
| iter() · next()<br>input() · print() | Python 내장 함수 | 반복자 · 다음 값<br>입력 · 출력 | `iter(values)` · `next(remaining)`<br>`print(f"{prompt}{value}")` |
| *values · 기본값<br>try · finally · yield<br>with · assert | Python 기본 문법 | 가변 인자 · 생략 가능 인자<br>마무리 보장 · 실행 넘김<br>시작 · 끝 자동 처리 · 검사 | `def fake_inputs(*values):`<br>`try: yield finally:`<br>`with fake_inputs("hello"):` |

---
## 1. 파이썬 기초

### 1-1. 3의 배수 출력

**한글 요약**

1부터 `n`까지의 수 중 **3의 배수만** 한 줄씩 출력. `range`의 간격을 3으로 지정해 나머지 검사 생략.

**실행 흐름**

1. **입력**: `n` = 7
2. **처리**: 3부터 n까지 3씩 증가하는 수 생성
3. **저장**: 없음 (바로 출력)
4. **출력**: 3의 배수 한 줄씩

In [2]:
# 함수 정의
def print_multiples_of_three(n: int) -> None:
    for i in range(3, n + 1, 3):  # 나머지 검사 없이 3의 배수만 만들기 위해 3씩 증가
        print(i)  # 문제 형식대로 한 줄에 하나씩 출력하기 위해

# 실행
print_multiples_of_three(7)  # 기대 결과 3, 6을 확인하기 위해

3
6


**결과 해설**

```
3, 6 : 7 이하의 3의 배수만 출력
```

**소속별 최종 요약표**

| 함수·문법 | 소속 | 역할 | 코드에서 사용한 위치 |
| :-- | :-- | :-- | :-- |
| print_multiples_of_three() | 사용자 정의 함수 | 3의 배수 출력 | `print_multiples_of_three(7)` |
| range() · print() | Python 내장 함수 | 간격 3 수열 · 출력 | `range(3, n + 1, 3)` · `print(i)` |
| def · for | Python 기본 문법 | 함수 정의 · 반복 | `def print_multiples_of_three(n: int) -> None:` |

### 1-2. 모음 제거하기

**한글 요약**

문자열에서 모음(a, e, i, o, u)을 제거해 반환. 대문자 모음까지 `set`에 담아 빠르게 포함 여부 확인.

**실행 흐름**

1. **입력**: 문자열 `"codeit"` · `"python programming"`
2. **처리**: 글자마다 모음 집합 포함 여부 검사 → 모음이 아닌 글자만 이어 붙이기
3. **저장**: `VOWELS` 모음 집합
4. **출력**: 모음을 뺀 문자열, 검증 결과

In [3]:
# 모음 집합
VOWELS = set("aeiouAEIOU")  # 대 · 소문자 모음을 한 번에 빠르게 검사하기 위해 set으로

# 함수 정의
def remove_vowels(s: str) -> str:
    return "".join(ch for ch in s if ch not in VOWELS)  # 모음이 아닌 글자만 남기기 위해

# 실행
print(remove_vowels("codeit"))  # 기대 결과 cdt를 확인하기 위해
print(remove_vowels("python programming"))  # 공백이 유지되는지 보기 위해

# 검증
assert remove_vowels("codeit") == "cdt"  # 기본 예시가 맞는지 확인하기 위해
assert remove_vowels("AEIOU") == ""  # 대문자 모음도 지워지는지 확인하기 위해
print("✅ 통과")  # 검증을 통과했음을 보이기 위해

cdt
pythn prgrmmng
✅ 통과


**결과 해설**

```
cdt            : codeit에서 o, e, i 제거
pythn prgrmmng : 공백은 모음이 아니므로 유지
✅ 통과        : 대문자 모음까지 제거 확인
```

**소속별 최종 요약표**

| 함수·문법 | 소속 | 역할 | 코드에서 사용한 위치 |
| :-- | :-- | :-- | :-- |
| remove_vowels() | 사용자 정의 함수 | 모음 제거 | `remove_vowels("codeit")` |
| join() | str | 글자들을 하나로 | `"".join(...)` |
| set() · print() | Python 내장 함수 | 모음 집합 · 출력 | `set("aeiouAEIOU")` |
| ( ... for ... if ... )<br>not in · assert | Python 기본 문법 | 조건에 맞는 글자만<br>미포함 검사 · 조건 검사 | `(ch for ch in s if ch not in VOWELS)`<br>`assert remove_vowels("AEIOU") == ""` |

### 1-3. 수 크기 비교하기

**한글 요약**

`(큰 수, 작은 수)` 반환. 두 수가 같으면 **그 수 하나만** 반환.

**실행 흐름**

1. **입력**: 두 수 (10, 20) · (30, 30)
2. **처리**: 같으면 그 수 하나, 다르면 `max` · `min`으로 순서 정리
3. **저장**: 없음
4. **출력**: 비교 결과, 검증 결과

In [4]:
# 함수 정의
def compare_numbers(num1: int, num2: int) -> tuple[int, int] | int:
    if num1 == num2:  # 문제 조건대로 같은 수는 하나만 반환하기 위해
        return num1
    return max(num1, num2), min(num1, num2)  # 입력 순서와 상관없이 (큰 수, 작은 수)가 되도록

# 실행
print(compare_numbers(10, 20))  # 작은 수가 앞에 올 때를 보기 위해
print(compare_numbers(30, 30))  # 같은 수일 때를 보기 위해

# 검증
assert compare_numbers(10, 20) == compare_numbers(20, 10) == (20, 10)  # 입력 순서와 무관한지 확인하기 위해
assert compare_numbers(30, 30) == 30  # 같은 수는 튜플이 아닌지 확인하기 위해
print("✅ 통과")  # 검증을 통과했음을 보이기 위해

(20, 10)
30
✅ 통과


**결과 해설**

```
(20, 10) : 작은 수가 앞에 와도 (큰 수, 작은 수) 순서
30       : 같은 수는 튜플이 아닌 숫자 하나
✅ 통과  : 입력 순서를 바꿔도 같은 결과
```

**소속별 최종 요약표**

| 함수·문법 | 소속 | 역할 | 코드에서 사용한 위치 |
| :-- | :-- | :-- | :-- |
| compare_numbers() | 사용자 정의 함수 | 두 수 비교 | `compare_numbers(10, 20)` |
| max() · min() · print() | Python 내장 함수 | 큰 수 · 작은 수 · 출력 | `max(num1, num2), min(num1, num2)` |
| if · 튜플 반환 · assert | Python 기본 문법 | 같은 수 판별 · 값 2개 반환 · 검사 | `return max(...), min(...)` |

### 1-4. 문자열 길이 반환하기

**한글 요약**

문자열의 길이 반환. 빈 문자열이면 `"문자열이 비어 있습니다."` 반환.

**실행 흐름**

1. **입력**: `"Python"` · 빈 문자열
2. **처리**: 비어 있으면 안내 문구, 아니면 `len()`
3. **저장**: 없음
4. **출력**: 길이 또는 안내 문구

In [5]:
# 함수 정의
def check_string_length(string: str) -> int | str:
    if not string:  # 빈 문자열은 False로 판단되므로 따로 처리하기 위해
        return "문자열이 비어 있습니다."
    return len(string)  # 글자 수를 돌려주기 위해

# 실행
print(check_string_length("Python"))  # 기대 결과 6을 확인하기 위해
print(check_string_length(""))  # 빈 문자열 안내를 확인하기 위해

# 검증
assert check_string_length("Python") == 6  # 길이가 맞는지 확인하기 위해
assert check_string_length("") == "문자열이 비어 있습니다."  # 안내 문구가 맞는지 확인하기 위해
print("✅ 통과")  # 검증을 통과했음을 보이기 위해

6
문자열이 비어 있습니다.
✅ 통과


**결과 해설**

```
6                       : "Python"의 글자 수
문자열이 비어 있습니다. : 빈 문자열은 길이 0 대신 안내 문구
✅ 통과                 : 두 경우의 반환값 확인
```

**소속별 최종 요약표**

| 함수·문법 | 소속 | 역할 | 코드에서 사용한 위치 |
| :-- | :-- | :-- | :-- |
| check_string_length() | 사용자 정의 함수 | 문자열 길이 확인 | `check_string_length("Python")` |
| len() · print() | Python 내장 함수 | 글자 수 · 출력 | `len(string)` |
| if not · assert | Python 기본 문법 | 빈 문자열 판별 · 검사 | `if not string:` |

### 1-5. 총 합계 구하기

**한글 요약**

0부터 `n`까지의 합 계산. 반복문 대신 **가우스 공식** `n(n+1)/2`로 O(1)에 계산하고, 반복문 결과와 비교해 검증.

**실행 흐름**

1. **입력**: `n` = 3 · 10
2. **처리**: `n × (n + 1) // 2` 계산 → 반복문 합과 비교 (n = 1 ~ 999)
3. **저장**: 없음
4. **출력**: 합계, 검증 결과

In [6]:
# 함수 정의
def sum_up_to_n(n: int) -> int:
    return n * (n + 1) // 2  # 반복 없이 한 번에 계산하기 위해 가우스 공식 (// 로 정수 유지)

# 실행
print(sum_up_to_n(3))  # 기대 결과 6을 확인하기 위해
print(sum_up_to_n(10))  # 기대 결과 55를 확인하기 위해

# 검증: 공식 결과 = 반복문으로 더한 결과
assert all(sum_up_to_n(n) == sum(range(n + 1)) for n in range(1, 1000))  # 공식이 맞는지 999가지 n으로 확인하기 위해
print("✅ 통과")  # 검증을 통과했음을 보이기 위해

6
55
✅ 통과


**결과 해설**

```
6       : 0 + 1 + 2 + 3
55      : 0 + 1 + … + 10
✅ 통과 : n = 1 ~ 999 전부 반복문 합과 일치
```

**소속별 최종 요약표**

| 함수·문법 | 소속 | 역할 | 코드에서 사용한 위치 |
| :-- | :-- | :-- | :-- |
| sum_up_to_n() | 사용자 정의 함수 | 0 ~ n 합계 | `sum_up_to_n(10)` |
| all() · sum() · range() | Python 내장 함수 | 모두 True 확인 · 비교용 합계 | `all(...)` · `sum(range(n + 1))` |
| // · ( ... for ... ) · assert | Python 기본 문법 | 정수 나눗셈 · n마다 비교 · 검사 | `n * (n + 1) // 2` |

### 1-6. 짝수 합 구하기

**한글 요약**

입력한 `n`까지의 짝수 합 출력, **0을 입력하면 종료**.

**실행 흐름**

1. **입력**: 자동 입력 9 → 0
2. **처리**: 0이면 종료, 아니면 2부터 n까지 2씩 더하기 → 다시 입력
3. **저장**: 없음
4. **출력**: 짝수 합 문장, 종료 문구

In [7]:
# 짝수 합 함수
def sum_even_numbers(n: int) -> None:  # 한 번의 입력을 처리하는 부분을 분리하기 위해
    if n == 0:  # 0을 종료 신호로 쓰기 위해
        print("종료")
        return
    print(f"1부터 {n}까지의 짝수 합은 {sum(range(2, n + 1, 2))}입니다.")  # 짝수만 더하기 위해 2부터 2씩 증가

# 반복 입력 함수
def even_sum_program() -> None:  # 반복 입력 흐름을 따로 묶기 위해
    """0이 입력될 때까지 반복"""
    while True:  # 0이 나올 때까지 계속 묻기 위해
        n = int(input("숫자를 입력하세요 (0: 종료): "))  # 계산하기 위해 문자열을 정수로
        sum_even_numbers(n)  # 입력마다 결과를 보이기 위해
        if n == 0:  # 0을 입력하면 반복을 끝내기 위해
            break

# 실행
with fake_inputs("9", "0"):  # 사람이 입력하지 않아도 실행되도록 9 → 0 자동 입력
    even_sum_program()  # 자동 입력으로 전체 흐름을 실행하기 위해

숫자를 입력하세요 (0: 종료): 9
1부터 9까지의 짝수 합은 20입니다.
숫자를 입력하세요 (0: 종료): 0
종료


**결과 해설**

```
9 → 20   : 2 + 4 + 6 + 8
0 → 종료 : 반복 종료
```

**소속별 최종 요약표**

| 함수·문법 | 소속 | 역할 | 코드에서 사용한 위치 |
| :-- | :-- | :-- | :-- |
| sum_even_numbers()<br>even_sum_program() | 사용자 정의 함수 | 짝수 합 출력<br>0까지 반복 입력 | `sum_even_numbers(n)`<br>`even_sum_program()` |
| int() · input()<br>sum() · range() | Python 내장 함수 | 정수 변환 · 입력<br>간격 2 합계 | `int(input(...))`<br>`sum(range(2, n + 1, 2))` |
| while True · break<br>return · f-string | Python 기본 문법 | 무한 반복 · 탈출<br>즉시 종료 · 값 넣은 문자열 | `while True:` · `break`<br>`f"1부터 {n}까지 ..."` |

### 1-7. 두 숫자 나누기

**한글 요약**

첫 번째 숫자를 두 번째 숫자로 나눈 값 반환. 숫자가 아니거나 0으로 나누면 `"잘못된 입력입니다."` 반환 (`try-except`).

<img src="images/m1_divide.png" alt="divide_numbers의 정상 경로와 예외 경로" width="800">

**실행 흐름**

1. **입력**: 인자 (42, 7) · (135, 0) · (7, 2), 인자 없으면 자동 입력
2. **처리**: 나눗셈 → 오류 3종류는 안내 문구 → 나누어떨어지면 int로
3. **저장**: 없음
4. **출력**: 몫 · 안내 문구, 검증 결과

In [8]:
# 함수 정의
def divide_numbers(a=None, b=None) -> int | float | str:
    """인자가 없으면 input()으로 두 숫자를 차례로 입력"""
    try:
        if a is None or b is None:  # 인자를 생략하면 직접 입력받기 위해
            a = float(input("첫 번째 숫자: "))  # 소수도 받기 위해 float으로
            b = float(input("두 번째 숫자: "))  # 소수도 받기 위해 float으로
        result = a / b  # 7 ÷ 2 = 3.5처럼 소수를 유지하기 위해 몫(//) 대신 일반 나눗셈
    except (ValueError, TypeError, ZeroDivisionError):  # 숫자 아님 · 잘못된 자료형 · 0으로 나누기를 한 번에 처리하기 위해
        return "잘못된 입력입니다."
    return int(result) if float(result).is_integer() else result  # 6.0을 6으로 보이기 위해

# 실행
print(divide_numbers(42, 7))  # 나누어떨어지는 경우를 보기 위해
print(divide_numbers(135, 0))  # 0으로 나누는 경우를 보기 위해
print(divide_numbers(7, 2))  # 소수가 남는 경우를 보기 위해
with fake_inputs("10", "zero"):  # 숫자가 아닌 입력을 재현하기 위해
    print(divide_numbers())

# 검증
assert divide_numbers(42, 7) == 6 and isinstance(divide_numbers(42, 7), int)  # 6.0이 아닌 정수 6인지 확인하기 위해
assert divide_numbers(7, 2) == 3.5  # 소수가 유지되는지 확인하기 위해
assert divide_numbers(135, 0) == divide_numbers("a", 1) == "잘못된 입력입니다."  # ZeroDivisionError · TypeError 처리를 확인하기 위해
print("✅ 통과")  # 검증을 통과했음을 보이기 위해

6
잘못된 입력입니다.
3.5
첫 번째 숫자: 10
두 번째 숫자: zero
잘못된 입력입니다.
✅ 통과


**결과 해설**

```
6                  : 42 ÷ 7 = 6.0 → 정수 6으로 표시
잘못된 입력입니다. : 0으로 나누기 (ZeroDivisionError)
3.5                : 나누어떨어지지 않으면 소수 그대로
zero 입력          : float() 변환 실패 (ValueError) → 안내 문구
✅ 통과            : 반환 자료형과 TypeError 처리까지 확인
```

**소속별 최종 요약표**

| 함수·문법 | 소속 | 역할 | 코드에서 사용한 위치 |
| :-- | :-- | :-- | :-- |
| divide_numbers() | 사용자 정의 함수 | 나눗셈 + 오류 처리 | `divide_numbers(42, 7)` |
| is_integer() | float | 소수점 아래가 0인지 | `float(result).is_integer()` |
| float() · int()<br>isinstance() | Python 내장 함수 | 실수 · 정수 변환<br>자료형 확인 | `float(input(...))`<br>`isinstance(..., int)` |
| 기본값 None · is None<br>try · except (…)<br>조건 표현식 | Python 기본 문법 | 인자 생략 허용 · 확인<br>오류 3종류 처리<br>한 줄 if-else | `def divide_numbers(a=None, b=None)`<br>`except (ValueError, TypeError, ZeroDivisionError):`<br>`int(result) if ... else result` |

---
## 2. 파이썬 응용

### 2-1. 문자열 압축

**한글 요약**

연속으로 반복되는 문자를 `문자 + 반복 횟수`로 압축. 예: `"aabcccccaaa"` → `"a2b1c5a3"`. 결과는 `itertools.groupby` 방식과 비교해 검증.

<img src="images/m1_compress.png" alt="aabcccccaaa가 a2b1c5a3으로 압축되는 과정" width="837">

**실행 흐름**

1. **입력**: `"aaabbaaa"` · `"aabcccccaaa"`
2. **처리**: 이웃한 두 글자 비교 → 같으면 개수 + 1, 다르면 조각 저장 → 마지막 묶음 추가
3. **저장**: 압축 조각 목록 `result`
4. **출력**: 압축 문자열, groupby와 비교한 검증 결과

In [9]:
# 모듈 불러오기
from itertools import groupby  # 다른 방법으로 구한 답과 비교하기 위해 (검증용)

# 함수 정의
def compress_string(s: str) -> str:
    if not s:  # 빈 문자열에서 s[-1] 오류가 나지 않도록
        return ""
    result, count = [], 1  # 압축 조각과 현재 글자 반복 수를 기록하기 위해
    for prev, cur in zip(s, s[1:]):  # 이웃한 두 글자를 비교하기 위해
        if cur == prev:  # 같은 글자가 이어지는지 확인하기 위해
            count += 1  # 반복 수를 세기 위해
        else:
            result.append(f"{prev}{count}")  # 묶음이 끝났으므로 "문자 + 개수"로 저장하기 위해
            count = 1  # 새 묶음을 1부터 세기 위해
    result.append(f"{s[-1]}{count}")  # 반복문 뒤에 남은 마지막 묶음을 넣기 위해
    return "".join(result)  # 조각을 하나의 문자열로 만들기 위해

# 실행
print(compress_string("aaabbaaa"))  # 떨어진 같은 글자를 따로 세는지 보기 위해
print(compress_string("aabcccccaaa"))  # 문제 예시를 확인하기 위해

# 검증: itertools.groupby로 구한 결과와 비교
for case in ["aaabbaaa", "aabcccccaaa", "", "a", "abc"]:  # 경계 사례까지 확인하기 위해
    assert compress_string(case) == "".join(f"{ch}{len(list(g))}" for ch, g in groupby(case))  # 두 방법의 결과가 같은지 확인하기 위해
print("✅ 통과")  # 검증을 통과했음을 보이기 위해

a3b2a3
a2b1c5a3
✅ 통과


**결과 해설**

```
a3b2a3   : 떨어져 있는 a 묶음은 따로 집계
a2b1c5a3 : 한 번만 나온 b도 1을 붙여 표시
✅ 통과  : 빈 문자열 · 한 글자 · 반복 없는 문자열까지 groupby 결과와 일치
```

**소속별 최종 요약표**

| 함수·문법 | 소속 | 역할 | 코드에서 사용한 위치 |
| :-- | :-- | :-- | :-- |
| compress_string() | 사용자 정의 함수 | 문자열 압축 | `compress_string("aabcccccaaa")` |
| groupby() | itertools | 연속된 같은 글자 묶기 | `groupby(case)` |
| append() · join() | list · str | 조각 추가 · 이어 붙이기 | `result.append(...)` · `"".join(result)` |
| zip() · len() · list() | Python 내장 함수 | 이웃 글자 짝 · 묶음 길이 | `zip(s, s[1:])` · `len(list(g))` |
| 슬라이싱 · 음수 인덱스<br>+= · assert | Python 기본 문법 | 두 번째부터 · 마지막 글자<br>개수 증가 · 검사 | `s[1:]` · `s[-1]`<br>`count += 1` |

### 2-2. 덧셈 프로그램

**한글 요약**

두 숫자를 입력받아 합 출력. 숫자가 아니면 `"올바른 숫자를 입력하세요."` 출력 후 **다시 입력**. 다시 묻는 부분은 `read_number()`로 분리해 2-3 · 5장에서 재사용.

**실행 흐름**

1. **입력**: 자동 입력 twenty → 10 → 20
2. **처리**: 정수가 될 때까지 다시 질문 → 두 수 더하기
3. **저장**: `read_number` 함수 (뒤 문제에서 재사용)
4. **출력**: 안내 문구, 덧셈 식

In [10]:
# 정수 입력 함수
def read_number(prompt: str) -> int:
    """정수가 입력될 때까지 다시 질문"""
    while True:  # 올바른 값이 나올 때까지 반복하기 위해
        try:
            return int(input(prompt))  # 변환에 성공하면 바로 반환해 반복을 끝내기 위해
        except ValueError:  # 숫자가 아닌 입력에도 프로그램이 멈추지 않도록
            print("올바른 숫자를 입력하세요.")  # 다시 입력하라고 안내하기 위해

# 덧셈 함수
def add_numbers() -> None:
    num1 = read_number("첫 번째 숫자를 입력하세요: ")  # 검증된 정수를 받기 위해
    num2 = read_number("두 번째 숫자를 입력하세요: ")  # 검증된 정수를 받기 위해
    print(f"{num1} + {num2} = {num1 + num2}")  # 식과 결과를 함께 보이기 위해

# 실행
with fake_inputs("twenty", "10", "20"):  # 잘못된 입력 → 다시 입력 흐름을 재현하기 위해
    add_numbers()

첫 번째 숫자를 입력하세요: twenty
올바른 숫자를 입력하세요.
첫 번째 숫자를 입력하세요: 10
두 번째 숫자를 입력하세요: 20
10 + 20 = 30


**결과 해설**

```
twenty       : 정수 변환 실패 → 안내 후 같은 질문 반복
10 + 20 = 30 : 올바른 두 숫자의 합
```

**소속별 최종 요약표**

| 함수·문법 | 소속 | 역할 | 코드에서 사용한 위치 |
| :-- | :-- | :-- | :-- |
| read_number()<br>add_numbers() | 사용자 정의 함수 | 정수가 나올 때까지 입력<br>두 수의 합 출력 | `read_number("첫 번째 ...")`<br>`add_numbers()` |
| int() · input() · print() | Python 내장 함수 | 정수 변환 · 입력 · 출력 | `int(input(prompt))` |
| while True<br>try · except ValueError | Python 기본 문법 | 올바른 입력까지 반복<br>변환 실패 처리 | `while True:`<br>`except ValueError:` |

### 2-3. 카운트다운 타이머

**한글 요약**

입력받은 시간(초)부터 1초마다 남은 시간 출력, 끝나면 `"타이머 종료!"` 출력. 음수는 안내 후 종료.

**실행 흐름**

1. **입력**: 자동 입력 5
2. **처리**: 음수면 안내 후 종료 → 남은 시간 출력 + 1초 대기 반복
3. **저장**: 없음
4. **출력**: 시작 · 남은 시간 · 종료 문구

In [11]:
# 모듈 불러오기
import time  # 1초씩 기다리기 위해

# 함수 정의
def countdown_timer(seconds: int) -> None:
    if seconds < 0:  # 음수 시간은 의미가 없어 거부하기 위해
        print("0 이상의 정수를 입력하세요.")
        return
    print("타이머 시작!")  # 시작을 알리기 위해
    for remaining in range(seconds, 0, -1):  # 무한 루프 없이 seconds부터 1까지 줄이기 위해
        print(remaining)  # 남은 시간을 보이기 위해
        time.sleep(1)  # 실제 1초 간격을 만들기 위해
    print("타이머 종료!")  # 끝났음을 알리기 위해

# 실행
with fake_inputs("5"):  # 사람이 입력하지 않아도 실행되도록
    countdown_timer(read_number("타이머를 시작할 시간(초)을 입력하세요: "))  # 잘못된 입력을 걸러내기 위해 2-2 함수 재사용

타이머를 시작할 시간(초)을 입력하세요: 5
타이머 시작!
5


4


3


2


1


타이머 종료!


**결과 해설**

```
5 → 1        : 1초 간격으로 남은 시간 출력 (총 약 5초)
타이머 종료! : 0이 되면 종료 문구
```

**소속별 최종 요약표**

| 함수·문법 | 소속 | 역할 | 코드에서 사용한 위치 |
| :-- | :-- | :-- | :-- |
| countdown_timer()<br>read_number() | 사용자 정의 함수 | 카운트다운<br>정수 입력 (2-2) | `countdown_timer(...)`<br>`read_number("타이머를 ...")` |
| sleep() | time | 1초 대기 | `time.sleep(1)` |
| range() · print() | Python 내장 함수 | 간격 -1 수열 · 출력 | `range(seconds, 0, -1)` |
| if · return · for | Python 기본 문법 | 음수 거부 · 남은 시간 반복 | `if seconds < 0:` |

---
## 3. 클래스 기초

### 3-1. 시간 추적 클래스 `TimeTracker`

**한글 요약**

`start()`로 시작, `stop()`으로 종료, `get_elapsed_time()`으로 경과 시간을 **분 단위**로 반환. 실제로 90분을 기다릴 수 없으므로 시작 시각을 1시간 30분 앞당겨 확인.

<img src="images/m1_timer.png" alt="TimeTracker로 90분 경과 시간을 확인하는 순서" width="912">

**실행 흐름**

1. **입력**: 현재 시각
2. **처리**: 시작 시각 기록 → 1시간 30분 앞당김 → 종료 시각 기록 → (종료 - 시작) 초 ÷ 60
3. **저장**: `study_session`의 시작 · 종료 시각
4. **출력**: 공부한 시간(분), 검증 결과

In [12]:
# 모듈 불러오기
from datetime import datetime, timedelta  # 현재 시각과 시간 간격을 다루기 위해

# 클래스 정의
class TimeTracker:
    def __init__(self) -> None:
        self.start_time: datetime | None = None  # 측정 전임을 None으로 구분하기 위해
        self.end_time: datetime | None = None  # 측정 전임을 None으로 구분하기 위해

    # 측정 시작 · 종료
    def start(self) -> None:
        self.start_time = datetime.now()  # 시작 시각을 기록하기 위해
        self.end_time = None  # 다시 시작하면 이전 종료 시각이 남지 않도록
    def stop(self) -> None:
        if self.start_time is None:  # 시작 전에 종료하는 실수를 막기 위해
            print("start()를 먼저 호출하세요.")
            return
        self.end_time = datetime.now()  # 종료 시각을 기록하기 위해

    # 경과 시간
    def get_elapsed_time(self) -> int:
        if self.start_time is None or self.end_time is None:  # 측정이 끝나지 않았으면 0분으로 처리하기 위해
            return 0
        return int((self.end_time - self.start_time).total_seconds() // 60)  # .seconds는 하루 넘는 시간을 잃으므로 total_seconds() 사용

# 실행
study_session = TimeTracker()  # 측정할 객체를 만들기 위해
study_session.start()  # 측정을 시작하기 위해
study_session.start_time -= timedelta(hours=1, minutes=30)  # 90분을 기다리지 않고 확인하기 위해 시작 시각을 앞당김
study_session.stop()  # 측정을 끝내기 위해
print("공부한 시간:", study_session.get_elapsed_time(), "분")  # 기대 결과 90분을 확인하기 위해

# 검증
assert study_session.get_elapsed_time() == 90  # 분 단위 계산이 맞는지 확인하기 위해
assert TimeTracker().get_elapsed_time() == 0  # 측정 전 객체가 0분인지 확인하기 위해
print("✅ 통과")  # 검증을 통과했음을 보이기 위해

공부한 시간: 90 분
✅ 통과


**결과 해설**

```
공부한 시간: 90 분 : 1시간 30분 = 90분
✅ 통과            : 측정 전 객체는 0분 반환
```

**소속별 최종 요약표**

| 함수·문법 | 소속 | 역할 | 코드에서 사용한 위치 |
| :-- | :-- | :-- | :-- |
| TimeTracker | 사용자 정의 클래스 | 경과 시간 측정 | `study_session = TimeTracker()` |
| datetime.now() · timedelta()<br>total_seconds() | datetime | 현재 시각 · 시간 간격<br>간격 → 초 | `datetime.now()` · `timedelta(hours=1, minutes=30)`<br>`(...).total_seconds()` |
| int() · print() | Python 내장 함수 | 분 단위 정수 · 출력 | `int(... // 60)` |
| class · \_\_init\_\_ · self<br>is None · -= · assert | Python 기본 문법 | 클래스 · 생성자 · 객체 자신<br>측정 여부 · 앞당기기 · 검사 | `class TimeTracker:`<br>`study_session.start_time -= timedelta(...)` |

### 3-2. 주소록 클래스 `Contact`

**한글 요약**

이름 · 전화번호 · 이메일 저장, `print()`하면 보기 좋게 출력. `__str__`로 출력 형식 지정.

**실행 흐름**

1. **입력**: 이름 · 전화번호 · 이메일
2. **처리**: 문자열로 저장 → `__str__`로 출력 형식 정의
3. **저장**: `friend` 객체
4. **출력**: 3줄 연락처

In [13]:
# 클래스 정의
class Contact:  # 연락처 정보를 한 객체로 묶기 위해
    def __init__(self, name: str, phone: str, email: str) -> None:  # 객체를 만들 때 세 정보를 받기 위해
        self.name, self.phone, self.email = str(name), str(phone), str(email)  # 숫자가 들어와도 같은 형식으로 출력되도록 문자열로 저장
    def __str__(self) -> str:  # print()가 이 형식을 쓰도록
        return f"이름: {self.name}\n전화번호: {self.phone}\n이메일: {self.email}"

# 실행
friend = Contact("Jane Doe", "010-1234-5678", "jane@example.com")  # 예시 연락처를 만들기 위해
print(friend)  # __str__에서 정한 형식을 확인하기 위해

이름: Jane Doe
전화번호: 010-1234-5678
이메일: jane@example.com


**결과 해설**

```
이름 · 전화번호 · 이메일 : __str__에서 정한 3줄 형식 그대로 출력
```

**소속별 최종 요약표**

| 함수·문법 | 소속 | 역할 | 코드에서 사용한 위치 |
| :-- | :-- | :-- | :-- |
| Contact | 사용자 정의 클래스 | 연락처 저장 · 출력 | `friend = Contact("Jane Doe", ...)` |
| str() · print() | Python 내장 함수 | 문자열 변환 · 출력 | `str(name)` · `print(friend)` |
| \_\_str\_\_ · 동시 대입 · \\n | Python 기본 문법 | 출력 형식 · 속성 3개 저장 · 줄 바꿈 | `def __str__(self) -> str:` |

---
## 4. 클래스 응용

### 4-1. 투표 시스템 `VoteSystem`

**한글 요약**

후보 등록 · 투표 · 결과 조회. **중복 등록과 미등록 후보 투표는 거부**, 득표는 딕셔너리로 관리.

**실행 흐름**

1. **입력**: 후보 3명, 투표 3회, 잘못된 시도 2건
2. **처리**: 등록 여부 검사 후 등록 · 투표 → 거부된 시도는 집계 제외
3. **저장**: `candidates` 딕셔너리 (후보 → 득표 수)
4. **출력**: 처리 안내, 투표 결과, 검증 결과

In [14]:
# 클래스 정의
class VoteSystem:
    def __init__(self) -> None:
        self.candidates: dict[str, int] = {}  # 투표마다 따로 관리하기 위해 인스턴스 변수로 (후보 → 득표 수)

    # 후보 등록
    def add_candidate(self, name: str) -> None:
        if name in self.candidates:  # 같은 후보가 두 번 등록되지 않도록
            print(f"{name} 후보는 이미 등록되어 있습니다.")
            return
        self.candidates[name] = 0  # 0표에서 시작하기 위해
        print(f"{name} 후보가 성공적으로 등록되었습니다.")  # 처리 결과를 알리기 위해

    # 투표
    def vote(self, name: str) -> None:
        if name not in self.candidates:  # 없는 후보 표가 집계되지 않도록
            print(f"{name} 후보는 등록되지 않아 투표할 수 없습니다.")
            return
        self.candidates[name] += 1  # 득표를 1 늘리기 위해
        print(f"{name}에게 투표하였습니다.")  # 처리 결과를 알리기 위해

    # 결과 조회
    def get_results(self) -> None:
        print("투표 결과:")  # 결과 구역을 구분하기 위해
        for name, votes in self.candidates.items():  # 등록 순서대로 보이기 위해
            print(f"{name}: {votes} {'vote' if votes == 1 else 'votes'}")  # 1표일 때만 단수형이 되도록

# 등록 · 투표
voting_system = VoteSystem()  # 투표 객체를 만들기 위해
for name in ["Alice", "Bob", "Charlie"]:  # 후보 3명을 등록하기 위해
    voting_system.add_candidate(name)
for name in ["Alice", "Alice", "Bob"]:  # 예시 투표 3회를 하기 위해
    voting_system.vote(name)

# 예외 상황
voting_system.add_candidate("Alice")  # 중복 등록이 거부되는지 보기 위해
voting_system.vote("Dave")  # 미등록 후보 투표가 거부되는지 보기 위해

# 결과 · 검증
voting_system.get_results()  # 최종 득표를 보기 위해
assert voting_system.candidates == {"Alice": 2, "Bob": 1, "Charlie": 0}  # 거부된 시도가 집계에서 빠졌는지 확인하기 위해
print("✅ 통과")  # 검증을 통과했음을 보이기 위해

Alice 후보가 성공적으로 등록되었습니다.
Bob 후보가 성공적으로 등록되었습니다.
Charlie 후보가 성공적으로 등록되었습니다.
Alice에게 투표하였습니다.
Alice에게 투표하였습니다.
Bob에게 투표하였습니다.
Alice 후보는 이미 등록되어 있습니다.
Dave 후보는 등록되지 않아 투표할 수 없습니다.
투표 결과:
Alice: 2 votes
Bob: 1 vote
Charlie: 0 votes
✅ 통과


**결과 해설**

```
등록 3건 · 투표 3건         : 정상 처리
Alice 중복 등록 · Dave 투표 : 안내 후 거부, 득표 변화 없음
Alice 2 · Bob 1 · Charlie 0 : 1표는 vote, 나머지는 votes
✅ 통과                     : 거부된 시도가 집계에 들어가지 않음
```

**소속별 최종 요약표**

| 함수·문법 | 소속 | 역할 | 코드에서 사용한 위치 |
| :-- | :-- | :-- | :-- |
| VoteSystem | 사용자 정의 클래스 | 후보 등록 · 투표 · 결과 | `voting_system = VoteSystem()` |
| items() | dict | (후보, 득표 수) 쌍 | `self.candidates.items()` |
| print() | Python 내장 함수 | 결과 출력 | `print(f"{name}: {votes} ...")` |
| in · not in · += 1<br>조건 표현식 · assert | Python 기본 문법 | 등록 여부 · 득표 증가<br>vote / votes · 검사 | `if name in self.candidates:`<br>`'vote' if votes == 1 else 'votes'` |

### 4-2. 은행 계좌 `BankAccount`

**한글 요약**

입금 · 출금 · 잔액 조회. **잔액보다 많이 출금하면 거부하고 경고**, 0 이하 · 숫자가 아닌 금액도 거부.

<img src="images/m1_bank.png" alt="입금 · 출금 5건과 잔액 변화, 거부된 2건" width="800">

**실행 흐름**

1. **입력**: 초기 잔액 10만 원, 입금 · 출금 4건
2. **처리**: 금액 검사 → 잔액 초과 출금 거부 → 정상 거래만 반영
3. **저장**: `_balance` (입금 · 출금 메서드로만 변경)
4. **출력**: 거래별 안내, 현재 잔액, 검증 결과

In [15]:
# 클래스 정의
class BankAccount:
    def __init__(self, account_number: str, account_holder: str, initial_balance: int | float = 0) -> None:
        if not self._is_valid_amount(initial_balance, allow_zero=True):  # 초기 잔액은 0도 허용하기 위해
            raise ValueError("초기 잔액은 0 이상의 숫자여야 합니다.")  # 잘못된 계좌가 만들어지지 않도록
        self.account_number = account_number  # 계좌를 구분하기 위해
        self.account_holder = account_holder  # 안내 문구에 쓰기 위해
        self._balance = initial_balance  # 밖에서 직접 바꾸지 않고 입금 · 출금으로만 바꾸기 위해
        print(f"{account_holder}님의 계좌 {account_number}가 개설되었습니다. 초기 잔액: {initial_balance}원")  # 개설 결과를 알리기 위해

    # 금액 검사
    @staticmethod
    def _is_valid_amount(amount, allow_zero: bool = False) -> bool:
        if isinstance(amount, bool) or not isinstance(amount, (int, float)):  # True/False · 문자열이 금액으로 쓰이지 않도록
            return False
        return amount >= 0 if allow_zero else amount > 0  # 상황에 따라 0 허용 여부를 바꾸기 위해

    # 입금 · 출금 · 조회
    def deposit(self, amount: int | float) -> None:
        if not self._is_valid_amount(amount):  # 0 이하 금액 입금을 막기 위해
            print(f"입금 실패: 0보다 큰 금액을 입력하세요. (입력: {amount!r})")
            return
        self._balance += amount  # 잔액을 늘리기 위해
        print(f"{amount}원이 입금되었습니다. 현재 잔액: {self._balance}원")  # 거래 결과를 알리기 위해
    def withdraw(self, amount: int | float) -> None:
        if not self._is_valid_amount(amount):  # 0 이하 금액 출금을 막기 위해
            print(f"출금 실패: 0보다 큰 금액을 입력하세요. (입력: {amount!r})")
        elif amount > self._balance:  # 잔액보다 많이 빠져나가지 않도록
            print(f"출금 실패: 잔액이 부족합니다. (요청 {amount}원, 잔액 {self._balance}원)")
        else:
            self._balance -= amount  # 잔액을 줄이기 위해
            print(f"{amount}원이 출금되었습니다. 현재 잔액: {self._balance}원")  # 거래 결과를 알리기 위해
    def get_balance(self) -> int | float:
        return self._balance  # 밖에서는 읽기만 하도록

# 실행
my_account = BankAccount("123-456-789", "김철수", 100000)  # 예시 계좌를 만들기 위해
my_account.deposit(50000)  # 정상 입금을 보기 위해
my_account.withdraw(20000)  # 정상 출금을 보기 위해
my_account.withdraw(1_000_000)  # 잔액 부족 거부를 보기 위해
my_account.deposit(-500)  # 잘못된 금액 거부를 보기 위해
print(f"현재 잔액: {my_account.get_balance()}원")  # 최종 잔액을 보기 위해

# 검증
assert my_account.get_balance() == 100000 + 50000 - 20000  # 거부된 거래가 잔액에 반영되지 않았는지 확인하기 위해
print("✅ 통과")  # 검증을 통과했음을 보이기 위해

김철수님의 계좌 123-456-789가 개설되었습니다. 초기 잔액: 100000원
50000원이 입금되었습니다. 현재 잔액: 150000원
20000원이 출금되었습니다. 현재 잔액: 130000원
출금 실패: 잔액이 부족합니다. (요청 1000000원, 잔액 130000원)
입금 실패: 0보다 큰 금액을 입력하세요. (입력: -500)
현재 잔액: 130000원
✅ 통과


**결과 해설**

```
100000 → 150000 → 130000 : 입금 5만 원, 출금 2만 원
1000000원 출금           : 잔액 13만 원보다 많아 거부
-500원 입금              : 0 이하 금액이라 거부
현재 잔액 130000원       : 거부된 두 거래는 반영되지 않음 (assert 통과)
```

**소속별 최종 요약표**

| 함수·문법 | 소속 | 역할 | 코드에서 사용한 위치 |
| :-- | :-- | :-- | :-- |
| BankAccount | 사용자 정의 클래스 | 입금 · 출금 · 잔액 관리 | `BankAccount("123-456-789", "김철수", 100000)` |
| isinstance() · print() | Python 내장 함수 | 숫자 자료형 확인 · 출력 | `isinstance(amount, (int, float))` |
| @staticmethod · raise<br>if · elif · else<br>_balance · !r · 1_000_000 | Python 기본 문법 | self 없는 메서드 · 오류 발생<br>금액 오류 · 잔액 부족 · 정상<br>직접 변경 금지 표시 · 따옴표 출력 · 자릿수 | `def _is_valid_amount(amount, ...)`<br>`elif amount > self._balance:`<br>`{amount!r}` · `1_000_000` |

### 4-3. 직원 관리 `EmployeeManager`

**한글 요약**

직원 정보를 **클래스 변수** `employees`에 모으고, **클래스 메서드**로 평균 급여 계산. 직원이 없으면 0 반환.

**실행 흐름**

1. **입력**: 직원 2명 (급여 50,000 · 60,000)
2. **처리**: 클래스 변수 초기화 → 생성자에서 자동 추가 → 클래스 메서드로 평균
3. **저장**: 클래스 변수 `employees`
4. **출력**: 추가 안내, 평균 급여, 검증 결과

In [16]:
# 클래스 정의
class EmployeeManager:
    employees: list["EmployeeManager"] = []  # 모든 인스턴스가 하나의 목록을 공유하도록 클래스 변수로
    def __init__(self, name: str, salary: int | float) -> None:
        self.name = name  # 직원을 구분하기 위해
        self.salary = salary  # 평균 급여 계산에 쓰기 위해
        EmployeeManager.employees.append(self)  # 동명이인도 따로 집계되도록 이름이 아닌 객체 저장
        print(f"{name} 님이 추가되었습니다. 급여: {salary}원")  # 추가 결과를 알리기 위해

    # 평균 급여
    @classmethod
    def calculate_average_salary(cls) -> float:
        if not cls.employees:  # 직원이 없을 때 0으로 나누지 않도록
            return 0
        average = sum(emp.salary for emp in cls.employees) / len(cls.employees)  # 전체 급여의 평균을 구하기 위해
        print(f"전체 직원의 평균 급여: {average:,.0f}원")  # 읽기 쉽도록 천 단위 쉼표
        return average

# 실행
EmployeeManager.employees.clear()  # 셀을 다시 실행해도 직원이 중복 저장되지 않도록
emp1 = EmployeeManager("홍길동", 50000)  # 예시 직원을 추가하기 위해
emp2 = EmployeeManager("김철수", 60000)  # 예시 직원을 추가하기 위해
average = EmployeeManager.calculate_average_salary()  # 객체 없이 클래스로 평균을 구하기 위해

# 검증
assert average == 55000 and len(EmployeeManager.employees) == 2  # 평균과 직원 수가 맞는지 확인하기 위해
print("✅ 통과")  # 검증을 통과했음을 보이기 위해

홍길동 님이 추가되었습니다. 급여: 50000원
김철수 님이 추가되었습니다. 급여: 60000원
전체 직원의 평균 급여: 55,000원
✅ 통과


**결과 해설**

```
직원 2명 추가      : 생성자에서 클래스 변수 employees에 자동 저장
평균 급여 55,000원 : (50,000 + 60,000) ÷ 2
✅ 통과            : 재실행해도 직원 수 2명 유지 (clear() 효과)
```

**소속별 최종 요약표**

| 함수·문법 | 소속 | 역할 | 코드에서 사용한 위치 |
| :-- | :-- | :-- | :-- |
| EmployeeManager | 사용자 정의 클래스 | 직원 목록 · 평균 급여 | `EmployeeManager("홍길동", 50000)` |
| append() · clear() | list | 직원 추가 · 목록 비우기 | `EmployeeManager.employees.clear()` |
| sum() · len() | Python 내장 함수 | 급여 합계 · 직원 수 | `sum(emp.salary for ...)` · `len(cls.employees)` |
| 클래스 변수 · @classmethod<br>:,.0f · assert | Python 기본 문법 | 공유 목록 · 클래스를 받는 메서드<br>천 단위 쉼표 · 검사 | `employees: list[...] = []`<br>`{average:,.0f}` |

### 4-4. 레스토랑 예약 `ReservationSystem` · 클래스

**한글 요약**

지점별 예약 추가 · 취소 · 조회와 여러 지점의 **전체 예약 수 합산**(클래스 메서드). 날짜는 3가지 표기를 `YYYY-MM-DD`로 통일하고, 잘못된 날짜 · 인원은 거부.

**실행 흐름**

1. **입력**: 지점 이름, 예약 (이름 · 날짜 · 인원)
2. **처리**: 날짜 표기를 하나씩 시도해 통일 → 인원 · 날짜 검사 → 예약 추가 · 취소 · 합산
3. **저장**: `ReservationSystem` 클래스
4. **출력**: 없음 (다음 셀에서 실행)

In [17]:
# 모듈 불러오기
from datetime import datetime  # 날짜 문자열을 해석 · 변환하기 위해

# 클래스 정의
class ReservationSystem:  # 지점마다 예약을 따로 관리하기 위해
    DATE_FORMATS = ("%Y-%m-%d", "%Y/%m/%d", "%Y.%m.%d")  # 허용할 날짜 표기 3가지를 한곳에 두기 위해
    def __init__(self, branch_name: str) -> None:  # 지점 이름으로 예약 객체를 만들기 위해
        self.branch_name = branch_name  # 안내 문구에 지점을 표시하기 위해
        self.reservations: list[dict] = []  # 예약 1건을 딕셔너리 1개로 보관하기 위해

    # 날짜 통일
    @classmethod  # 객체 없이 클래스의 DATE_FORMATS를 쓰기 위해
    def _normalize_date(cls, date: str) -> str:  # 표기가 달라도 같은 날짜로 비교하기 위해
        """2024-05-20 · 2024/05/20 · 2024.05.20 → YYYY-MM-DD"""
        for fmt in cls.DATE_FORMATS:  # 표기를 하나씩 시도하기 위해
            try:  # 표기가 맞지 않으면 다음 표기로 넘어가기 위해
                return datetime.strptime(str(date).strip(), fmt).strftime("%Y-%m-%d")  # 어떤 표기든 같은 형식으로 저장하기 위해
            except ValueError:  # 이 표기가 아니면 다음 표기를 시도하기 위해
                continue
        raise ValueError(f"날짜 형식이 올바르지 않습니다: {date!r}")  # 3가지 모두 실패하면 알리기 위해

    # 예약 추가 · 취소 · 조회
    def add_reservation(self, name: str, date: str, guests: int) -> None:  # 검사를 통과한 예약만 저장하기 위해
        try:  # 인원 · 날짜 오류를 한곳에서 처리하기 위해
            if not isinstance(guests, int) or guests <= 0:  # 인원은 1 이상의 정수만 받기 위해
                raise ValueError(f"인원 수는 1 이상의 정수여야 합니다: {guests!r}")
            date = self._normalize_date(date)  # 날짜 표기를 통일하기 위해
        except ValueError as e:  # 인원 · 날짜 오류를 한곳에서 안내하기 위해
            print(f"[{self.branch_name}] 예약 실패 - {e}")
            return
        self.reservations.append({"name": name, "date": date, "guests": guests})  # 검사를 통과한 예약만 저장하기 위해
    def cancel_reservation(self, name: str, date: str) -> None:  # 이름 · 날짜로 예약을 취소하기 위해
        for r in self.reservations:  # 취소할 예약을 찾기 위해
            if r["name"] == name and r["date"] == date:  # 이름과 날짜가 모두 같은 예약만 취소하기 위해
                self.reservations.remove(r)  # 목록에서 지우기 위해
                print(f"[{self.branch_name}] {name}님의 {date} 예약이 취소되었습니다.")
                return
        print(f"[{self.branch_name}] 취소 실패 - {name}님의 {date} 예약이 없습니다.")  # 없는 예약임을 알리기 위해
    def list_reservations(self) -> None:  # 지점별 예약을 확인하기 위해
        print(f"{self.branch_name} 예약 목록:")  # 지점별로 구분하기 위해
        for r in self.reservations:  # 저장 순서대로 보이기 위해
            print(f"- {r['name']}, {r['date']}, {r['guests']}명")

    # 전체 예약 수
    @classmethod  # 여러 지점을 객체 없이 합산하기 위해
    def sum_reservations(cls, branches: list["ReservationSystem"]) -> int:  # 전체 예약 수를 구하기 위해
        return sum(len(branch.reservations) for branch in branches)  # 지점 수가 아닌 예약 건수를 더하기 위해

**결과 해설**

```
출력 없음 : 클래스 정의 셀, 다음 셀에서 두 지점으로 실행
```

**소속별 최종 요약표**

| 함수·문법 | 소속 | 역할 | 코드에서 사용한 위치 |
| :-- | :-- | :-- | :-- |
| ReservationSystem | 사용자 정의 클래스 | 지점별 예약 관리 · 합산 | `ReservationSystem("강남점")` |
| strptime() · strftime() | datetime | 문자열 ↔ 날짜 | `datetime.strptime(..., fmt).strftime("%Y-%m-%d")` |
| append() · remove() · strip() | list · str | 예약 추가 · 삭제 · 공백 제거 | `self.reservations.remove(r)` |
| isinstance() · sum() · len() | Python 내장 함수 | 인원 자료형 · 예약 수 합계 | `sum(len(branch.reservations) for ...)` |
| @classmethod · continue<br>except ValueError as e · raise | Python 기본 문법 | 클래스 기준 메서드 · 다음 표기<br>오류 받아 안내 · 오류 발생 | `def sum_reservations(cls, branches)`<br>`except ValueError as e:` |

### 4-4. 레스토랑 예약 `ReservationSystem` · 실행

**한글 요약**

지점 2개에 예약 4건을 시도해 날짜 표기 통일 · 잘못된 날짜 거부 · 전체 예약 수 합산을 확인.

**실행 흐름**

1. **입력**: 강남점 1건, 홍대점 3건 (그중 1건은 잘못된 날짜)
2. **처리**: 예약 추가 → 지점별 목록 → 전체 예약 수 합산
3. **저장**: `restaurant1` · `restaurant2` · `total_reservations`
4. **출력**: 거부 안내, 지점별 목록, 전체 예약 수, 검증 결과

In [18]:
# 예약 추가
restaurant1 = ReservationSystem("강남점")  # 지점마다 따로 관리하기 위해
restaurant2 = ReservationSystem("홍대점")  # 지점마다 따로 관리하기 위해
restaurant1.add_reservation("홍길동", "2024-05-20", 4)  # 정상 예약을 넣기 위해
restaurant2.add_reservation("김철수", "2024-05-21", 2)  # 정상 예약을 넣기 위해
restaurant2.add_reservation("이영희", "2024/05/22", 3)  # 다른 날짜 표기가 통일되는지 보기 위해
restaurant2.add_reservation("박민수", "5월 23일", 2)  # 잘못된 날짜가 거부되는지 보기 위해

# 조회 · 합산
restaurant1.list_reservations()  # 지점별 예약을 확인하기 위해
restaurant2.list_reservations()  # 지점별 예약을 확인하기 위해
total_reservations = ReservationSystem.sum_reservations([restaurant1, restaurant2])  # 여러 지점의 예약 수를 합치기 위해
print(f"전체 레스토랑 예약 수: {total_reservations}")  # 기대 결과 3을 확인하기 위해

# 검증
assert total_reservations == 3  # 거부된 예약이 합산에서 빠졌는지 확인하기 위해
assert restaurant2.reservations[1]["date"] == "2024-05-22"  # 2024/05/22가 YYYY-MM-DD로 통일됐는지 확인하기 위해
print("✅ 통과")  # 검증을 통과했음을 보이기 위해

[홍대점] 예약 실패 - 날짜 형식이 올바르지 않습니다: '5월 23일'
강남점 예약 목록:
- 홍길동, 2024-05-20, 4명
홍대점 예약 목록:
- 김철수, 2024-05-21, 2명
- 이영희, 2024-05-22, 3명
전체 레스토랑 예약 수: 3
✅ 통과


**결과 해설**

```
[홍대점] 예약 실패      : "5월 23일"은 허용 표기 3가지에 해당하지 않음
이영희, 2024-05-22      : 2024/05/22 표기가 YYYY-MM-DD로 통일
전체 레스토랑 예약 수 3 : 강남점 1건 + 홍대점 2건 (거부된 1건 제외)
```

**소속별 최종 요약표**

| 함수·문법 | 소속 | 역할 | 코드에서 사용한 위치 |
| :-- | :-- | :-- | :-- |
| ReservationSystem | 사용자 정의 클래스 | 예약 추가 · 조회 · 합산 | `restaurant2.add_reservation("이영희", "2024/05/22", 3)` |
| print() | Python 내장 함수 | 전체 예약 수 출력 | `print(f"전체 레스토랑 예약 수: ...")` |
| 클래스 메서드 호출 · assert | Python 기본 문법 | 객체 없이 클래스로 합산 · 검사 | `ReservationSystem.sum_reservations([...])` |

---
## 5. 심화

### 5-1. 시간 관리 프로그램 · 함수

**한글 요약**

CSV 파일의 할 일 목록에서 **남은 시간 안에 할 수 있는 일**을 골라 소요 시간이 짧은 순으로 출력.

- **기준 1 · 할 일 개수 최대**: 짧은 일부터 담는 그리디 방식으로 최대 개수 `k` 계산
- **기준 2 · 남는 시간 최소**: 개수가 `k`인 조합이 여럿이면 시간을 가장 많이 쓰는 조합 선택

**실행 흐름**

1. **입력**: 예시 할 일 4개
2. **처리**: 예시 CSV 생성 → 불러오기 함수 → 기준 1 · 2로 고르는 함수 → 입력 · 출력 함수
3. **저장**: `tasks.csv`, 함수 3개
4. **출력**: 없음 (다음 셀에서 실행)

In [19]:
# 모듈 불러오기
import csv  # CSV 파일을 읽고 쓰기 위해
from itertools import combinations  # 개수가 k인 조합을 모두 만들기 위해

# 예시 파일 생성
sample_tasks = [("공부하기", 120), ("운동하기", 60), ("가계부정리", 30), ("영화보기", 150)]  # 문제 예시를 재현하기 위해
with open("tasks.csv", "w", newline="", encoding="utf-8") as f:  # Windows에서도 한글이 깨지지 않도록 utf-8
    writer = csv.writer(f)  # CSV 형식으로 쓰기 위해
    writer.writerow(["할일", "소요시간"])  # 열 이름으로 읽을 수 있도록 머리글
    writer.writerows(sample_tasks)  # 할 일을 한 번에 쓰기 위해

# 함수 정의
def load_tasks(filename: str) -> list[tuple[str, int]]:
    with open(filename, newline="", encoding="utf-8") as f:  # 쓸 때와 같은 인코딩으로 읽기 위해
        return [(row["할일"], int(row["소요시간"])) for row in csv.DictReader(f)]  # 열 순서가 바뀌어도 읽히도록 머리글 이름 사용
def suggest_tasks(tasks: list[tuple[str, int]], remaining_time: int) -> list[tuple[str, int]]:
    k, used = 0, 0  # 기준 1을 위해 담은 개수와 사용 시간 기록
    for _, duration in sorted(tasks, key=lambda t: t[1]):  # 개수를 최대로 하기 위해 짧은 일부터 담음
        if used + duration > remaining_time:  # 남은 시간을 넘기지 않도록
            break
        used += duration  # 사용 시간을 누적하기 위해
        k += 1  # 담은 개수를 세기 위해
    if k == 0:  # 가장 짧은 일도 못 하면 빈 목록을 주기 위해
        return []
    feasible = [c for c in combinations(tasks, k) if sum(d for _, d in c) <= remaining_time]  # 기준 2를 위해 시간 안에 드는 k개 조합만
    best = max(feasible, key=lambda c: sum(d for _, d in c))  # 남는 시간이 가장 적도록 시간을 가장 많이 쓰는 조합
    return sorted(best, key=lambda t: t[1])  # 문제 형식대로 짧은 순 정렬
def main() -> None:
    filename = input("CSV 파일 경로: ").strip() or "tasks.csv"  # 빈 입력이면 기본 파일을 쓰기 위해
    try:
        tasks = load_tasks(filename)  # 할 일 목록을 읽기 위해
    except FileNotFoundError:  # 없는 파일에도 프로그램이 멈추지 않도록
        print(f"파일을 찾을 수 없습니다: {filename}")
        return
    remaining_time = read_number("남은 시간(분): ")  # 잘못된 입력을 걸러내기 위해 2-2 함수 재사용
    selected = suggest_tasks(tasks, remaining_time)  # 기준 1 · 2로 할 일을 고르기 위해
    if not selected:  # 할 수 있는 일이 없을 때 안내하기 위해
        print("남은 시간 안에 할 수 있는 일이 없습니다.")
        return
    print("선택된 할 일 목록:")  # 결과 구역을 구분하기 위해
    for i, (name, duration) in enumerate(selected, start=1):  # 1번부터 번호를 매기기 위해
        print(f"{i}. {name} - 예상 소요 시간: {duration}분")

**결과 해설**

```
출력 없음 : 예시 CSV 생성과 함수 정의, 다음 셀에서 남은 시간 210분 · 200분으로 실행
```

**소속별 최종 요약표**

| 함수·문법 | 소속 | 역할 | 코드에서 사용한 위치 |
| :-- | :-- | :-- | :-- |
| load_tasks() · suggest_tasks() · main() | 사용자 정의 함수 | CSV 읽기 · 기준 1 · 2 선택 · 입출력 | `suggest_tasks(tasks, 200)` |
| writer() · writerow(s) · DictReader() | csv | CSV 쓰기 · 머리글로 읽기 | `csv.DictReader(f)` |
| combinations() | itertools | k개 조합 | `combinations(tasks, k)` |
| open() · sorted() · max()<br>enumerate() | Python 내장 함수 | 파일 열기 · 정렬 · 최댓값<br>번호 매기기 | `sorted(tasks, key=lambda t: t[1])`<br>`enumerate(selected, start=1)` |
| with · lambda<br>[ ... for ... if ... ] · or | Python 기본 문법 | 파일 자동 닫기 · 정렬 기준<br>조건 조합 · 빈 입력 기본값 | `key=lambda c: sum(...)`<br>`... .strip() or "tasks.csv"` |

### 5-1. 시간 관리 프로그램 · 실행

**한글 요약**

남은 시간 210분 · 200분으로 실행. 200분일 때 **기준 2**가 그리디 결과(90분)보다 시간을 잘 쓰는 조합(180분)을 고르는지 확인.

<img src="images/m1_tasks.png" alt="200분일 때 그리디 조합과 기준 2 조합 비교" width="837">

**실행 흐름**

1. **입력**: 자동 입력 (파일 경로, 남은 시간 210 · 200)
2. **처리**: `main()` 두 번 실행 → 기준 2 효과와 불가능한 경우 검증
3. **저장**: `tasks`
4. **출력**: 선택된 할 일 목록 2개, 검증 결과

In [20]:
# 실행: 남은 시간 210분
with fake_inputs("tasks.csv", "210"):  # 사람이 입력하지 않아도 실행되도록
    main()

# 실행: 남은 시간 200분 (기준 2 효과 확인)
with fake_inputs("tasks.csv", "200"):  # 기준 2가 다른 답을 고르는 경우를 보기 위해
    main()

# 검증
tasks = load_tasks("tasks.csv")  # 함수를 직접 검증하기 위해 다시 읽음
assert [name for name, _ in suggest_tasks(tasks, 200)] == ["운동하기", "공부하기"]  # 그리디(90분) 대신 180분 조합을 고르는지 확인하기 위해
assert suggest_tasks(tasks, 10) == []  # 가장 짧은 일(30분)보다 시간이 적으면 빈 목록인지 확인하기 위해
print("✅ 통과")  # 검증을 통과했음을 보이기 위해

CSV 파일 경로: tasks.csv
남은 시간(분): 210
선택된 할 일 목록:
1. 가계부정리 - 예상 소요 시간: 30분
2. 운동하기 - 예상 소요 시간: 60분
3. 공부하기 - 예상 소요 시간: 120분
CSV 파일 경로: tasks.csv
남은 시간(분): 200
선택된 할 일 목록:
1. 운동하기 - 예상 소요 시간: 60분
2. 공부하기 - 예상 소요 시간: 120분
✅ 통과


**결과 해설**

```
210분 → 3개 (30 + 60 + 120 = 210분) : 최대 개수 3개, 남는 시간 0분
200분 → 2개 (60 + 120 = 180분)      : 최대 개수 2개, 시간을 가장 많이 쓰는 조합
그리디만 썼다면                     : 가계부정리 + 운동하기(90분) → 남는 시간 110분 (기준 2 적용 시 20분)
✅ 통과                             : 기준 2 적용 · 불가능한 경우 빈 목록 확인
```

**소속별 최종 요약표**

| 함수·문법 | 소속 | 역할 | 코드에서 사용한 위치 |
| :-- | :-- | :-- | :-- |
| main() · load_tasks() · suggest_tasks() | 사용자 정의 함수 | 입출력 · 읽기 · 선택 | `main()` · `suggest_tasks(tasks, 10)` |
| fake_inputs() | 사용자 정의 함수 | 입력값 자동 공급 (0-1) | `with fake_inputs("tasks.csv", "210"):` |
| [ ... for ... ] · assert | Python 기본 문법 | 이름만 꺼내기 · 검사 | `[name for name, _ in suggest_tasks(...)]` |

### 5-2. 수학 퍼즐: 소수의 합

**한글 요약**

입력한 수 이하의 소수를 모두 더하기. **에라토스테네스의 체**로 계산하고, 하나씩 판별해 더한 값과 비교해 검증.

**실행 흐름**

1. **입력**: 자동 입력 10
2. **처리**: 0 ~ n을 소수 후보로 표시 → 2부터 √n까지 소수의 배수 지우기 → 남은 수 합계
3. **저장**: 체 목록 `sieve`
4. **출력**: 소수 합 문장, 하나씩 판별한 합과 비교한 검증 결과

In [21]:
# 소수 판별 함수 (검증용)
def is_prime(num: int) -> bool:
    """√num까지만 나눠 보면 소수 여부 확인 가능"""
    if num < 2:  # 0 · 1은 소수가 아니므로 먼저 걸러내기 위해
        return False
    return all(num % i for i in range(2, int(num ** 0.5) + 1))  # 계산을 줄이기 위해 √num까지만 나눠 봄

# 에라토스테네스의 체
def sum_of_primes(n: int) -> int:
    """에라토스테네스의 체: O(n log log n)"""
    if n < 2:  # 소수가 없는 범위를 바로 처리하기 위해
        return 0
    sieve = [True] * (n + 1)  # 인덱스를 수로 쓰기 위해 n + 1칸 (True = 소수 후보)
    sieve[0] = sieve[1] = False  # 0 · 1은 소수가 아니므로 지우기 위해
    for i in range(2, int(n ** 0.5) + 1):  # √n 이후 배수는 이미 지워졌기 때문에 √n까지만
        if sieve[i]:  # 아직 남아 있으면 소수이므로 배수를 지우기 위해
            sieve[i * i :: i] = [False] * len(range(i * i, n + 1, i))  # i의 배수를 한 번에 지우기 위해
    return sum(i for i, prime in enumerate(sieve) if prime)  # 남은 소수만 더하기 위해

# 실행
def main() -> None:
    n = read_number("어떤 수까지의 소수의 합을 구하시겠습니까?: ")  # 잘못된 입력을 걸러내기 위해 2-2 함수 재사용
    print(f"{n} 이하 소수의 합은 {sum_of_primes(n)}입니다.")  # 문제 형식대로 출력하기 위해
with fake_inputs("10"):  # 사람이 입력하지 않아도 실행되도록
    main()  # 기대 결과 17을 확인하기 위해

# 검증: 체로 구한 합 = 하나씩 판별해 더한 합
assert all(sum_of_primes(n) == sum(x for x in range(n + 1) if is_prime(x)) for n in range(500))  # 두 방법이 500가지 n에서 같은지 확인하기 위해
print("✅ 통과")  # 검증을 통과했음을 보이기 위해

어떤 수까지의 소수의 합을 구하시겠습니까?: 10
10 이하 소수의 합은 17입니다.
✅ 통과


**결과 해설**

```
10 이하 소수의 합 17 : 2 + 3 + 5 + 7
✅ 통과              : n = 0 ~ 499 전부 하나씩 판별한 합과 일치
```

**소속별 최종 요약표**

| 함수·문법 | 소속 | 역할 | 코드에서 사용한 위치 |
| :-- | :-- | :-- | :-- |
| is_prime() · sum_of_primes() · main() | 사용자 정의 함수 | 소수 판별 · 체로 합계 · 입출력 | `sum_of_primes(n)` |
| all() · range() · len()<br>enumerate() · sum() · int() | Python 내장 함수 | 나누어떨어짐 확인 · 배수 개수<br>(수, 소수 여부) · 합 · √n 정수화 | `all(num % i for i in ...)`<br>`sum(i for i, prime in enumerate(sieve) if prime)` |
| [True] * (n + 1) · 슬라이싱 대입<br>** 0.5 · assert | Python 기본 문법 | 같은 값 목록 · 배수 한 번에 지우기<br>제곱근 · 검사 | `sieve[i * i :: i] = [False] * ...`<br>`num ** 0.5` |

### 5-3. 도서관 관리 시스템 · 예외와 정보 클래스

**한글 요약**

도서 추가 · 삭제 · 검색, 회원 등록, 대여 · 반납 관리. 도서 · 회원이 없을 때 등 **오류 상황마다 알맞은 안내**. 먼저 오류 종류(사용자 정의 예외)와 도서 · 회원 · 대여 기록 클래스 정의.

<img src="images/m1_library.png" alt="예외 5개 · 정보 클래스 3개 · 관리 클래스 2개의 관계" width="837">

**실행 흐름**

1. **입력**: 없음
2. **처리**: 오류 종류별 예외 클래스 → 예외를 안내 문구로 바꾸는 데코레이터 → `Book` · `Member` · `Rental`
3. **저장**: 예외 5개 · `report_errors` · 정보 클래스 3개
4. **출력**: 없음 (정의 셀)

In [22]:
# 모듈 불러오기
from datetime import date, timedelta  # 대여일 · 반납 예정일을 계산하기 위해
from functools import wraps  # 데코레이터를 씌워도 원래 메서드 이름이 유지되도록

# 사용자 정의 예외
class LibraryError(Exception):  # 도서관 오류만 골라 처리하기 위해 공통 부모
    """도서관 예외의 부모 클래스"""
class BookNotFoundError(LibraryError):  # 없는 도서를 따로 구분하기 위해
    def __init__(self, isbn):  # 어떤 도서인지 안내에 담기 위해
        super().__init__(f"ISBN '{isbn}' 도서를 찾을 수 없습니다.")  # 안내 문구를 예외에 담기 위해
class BookNotAvailableError(LibraryError):  # 재고 없음을 따로 구분하기 위해
    def __init__(self, title):  # 어떤 도서인지 안내에 담기 위해
        super().__init__(f"'{title}' 도서는 모두 대여 중입니다.")  # 안내 문구를 예외에 담기 위해
class MemberNotFoundError(LibraryError):  # 없는 회원을 따로 구분하기 위해
    def __init__(self, name):  # 어떤 회원인지 안내에 담기 위해
        super().__init__(f"회원 '{name}'님을 찾을 수 없습니다.")  # 안내 문구를 예외에 담기 위해
class NotRentedError(LibraryError):  # 빌리지 않은 책 반납을 따로 구분하기 위해
    def __init__(self, name, title):  # 회원과 도서를 함께 안내하기 위해
        super().__init__(f"'{name}' 회원님은 '{title}' 도서를 대여하고 있지 않습니다.")  # 안내 문구를 예외에 담기 위해

# 오류 안내 데코레이터
def report_errors(method):  # 메서드마다 try를 반복하지 않도록
    """실행 중 생긴 LibraryError를 '오류: ...'로 안내"""
    @wraps(method)  # 감싼 뒤에도 원래 메서드 이름이 보이도록
    def wrapper(*args, **kwargs):  # 어떤 메서드에도 씌울 수 있도록 인자를 그대로 전달
        try:  # 도서관 오류를 잡아 안내하기 위해
            return method(*args, **kwargs)  # 원래 메서드를 실행하기 위해
        except LibraryError as e:  # 도서관 오류만 안내하고 다른 오류는 그대로 드러나도록
            print(f"오류: {e}")
    return wrapper  # 원래 메서드 대신 감싼 함수를 쓰도록

**결과 해설**

```
출력 없음 : 오류 종류 5개와 안내 데코레이터 정의, 5-3 관리 클래스에서 사용
```

**소속별 최종 요약표**

| 함수·문법 | 소속 | 역할 | 코드에서 사용한 위치 |
| :-- | :-- | :-- | :-- |
| LibraryError 외 4개 예외 | 사용자 정의 클래스 | 오류 종류 구분 | `class BookNotFoundError(LibraryError):` |
| report_errors() | 사용자 정의 함수 | LibraryError를 안내 문구로 | `@report_errors` |
| @wraps | functools | 원래 메서드 이름 유지 | `@wraps(method)` |
| 상속 · super()<br>*args · **kwargs | Python 기본 문법 | 예외 상속 · 부모 생성자<br>인자 그대로 전달 | `super().__init__(...)`<br>`def wrapper(*args, **kwargs):` |

### 5-3. 도서관 관리 시스템 · 정보 클래스

**한글 요약**

도서(`Book`) · 회원(`Member`) · 대여 기록(`Rental`) 정보를 담는 클래스. 반납 예정일은 대여일 + 14일.

**실행 흐름**

1. **입력**: 없음
2. **처리**: 도서 정보 · 권수 → 회원 · 대여 목록 → 대여일 · 반납 예정일 · 반납일
3. **저장**: `Book` · `Member` · `Rental` 클래스
4. **출력**: 없음 (정의 셀)

In [23]:
# 도서 · 회원 · 대여 기록
class Book:  # 도서 정보와 권수를 한 객체로 묶기 위해
    def __init__(self, title: str, author: str, publication_year: int, isbn: str) -> None:  # 도서 정보를 받아 저장하기 위해
        self.title, self.author = title, author  # 검색 · 출력에 쓰기 위해
        self.publication_year, self.isbn = publication_year, isbn  # 출력 · 식별에 쓰기 위해
        self.total_copies = self.available_copies = 1  # 보유 권수와 대여 가능 권수를 따로 세기 위해
    def __str__(self) -> str:  # print()로 도서를 보기 좋게 출력하기 위해
        return f"{self.title} (저자: {self.author}, 출판년도: {self.publication_year})"
class Member:  # 회원과 대여 목록을 한 객체로 묶기 위해
    def __init__(self, name: str, member_id: int) -> None:  # 회원 정보를 받아 저장하기 위해
        self.name, self.member_id = name, member_id  # 회원을 구분하기 위해
        self.rented_books: list[Book] = []  # 대여 중인 도서를 보이기 위해
    def __str__(self) -> str:  # print()로 회원 상태를 보기 좋게 출력하기 위해
        titles = ", ".join(b.title for b in self.rented_books) or "없음"  # 대여 도서가 없을 때 "없음"으로 보이기 위해
        return f"{self.name} (대여 중인 도서: {titles})"
class Rental:  # 대여 한 건의 날짜를 기록하기 위해
    LOAN_DAYS = 14  # 대여 기간을 한곳에서 바꿀 수 있도록
    def __init__(self, book: Book, member: Member) -> None:  # 누가 무엇을 빌렸는지 받기 위해
        self.book, self.member = book, member  # 누가 무엇을 빌렸는지 기록하기 위해
        self.rental_date = date.today()  # 대여일을 기록하기 위해
        self.due_date = self.rental_date + timedelta(days=self.LOAN_DAYS)  # 반납 예정일을 계산하기 위해
        self.return_date: date | None = None  # 반납 전임을 None으로 구분하기 위해

**결과 해설**

```
출력 없음 : 도서 · 회원 · 대여 기록 클래스 정의, 다음 셀의 관리 클래스가 사용
```

**소속별 최종 요약표**

| 함수·문법 | 소속 | 역할 | 코드에서 사용한 위치 |
| :-- | :-- | :-- | :-- |
| Book · Member · Rental | 사용자 정의 클래스 | 도서 · 회원 · 대여 기록 정보 | `Rental(book, member)` |
| date.today() · timedelta() | datetime | 오늘 날짜 · 14일 간격 | `timedelta(days=self.LOAN_DAYS)` |
| join() | str | 대여 도서 제목 연결 | `", ".join(...)` |
| \_\_str\_\_ · 클래스 변수 · or | Python 기본 문법 | 출력 형식 · 대여 기간 · 빈 목록 기본값 | `LOAN_DAYS = 14` · `... or "없음"` |

### 5-3. 도서관 관리 시스템 · 도서 관리 클래스

**한글 요약**

도서 추가 · 삭제 · 검색과 조회 도우미를 `LibraryCatalog`에 정의. 찾지 못하면 예외를 던지고, `@report_errors`가 안내 문구로 바꿈.

**실행 흐름**

1. **입력**: 없음
2. **처리**: 도서 · 회원 조회 도우미 → 같은 도서는 권수만 증가 → 대여 중이면 삭제 거부 → 제목 · 저자 검색
3. **저장**: `LibraryCatalog` 클래스
4. **출력**: 없음 (정의 셀)

In [24]:
# 도서 관리
class LibraryCatalog:  # 도서 관리 기능을 따로 묶기 위해
    def __init__(self) -> None:  # 빈 도서관에서 시작하기 위해
        self.books: dict[str, Book] = {}  # ISBN으로 바로 찾기 위해 (ISBN → Book)
        self.members: dict[str, Member] = {}  # 이름으로 바로 찾기 위해 (이름 → Member)
        self.rentals: list[Rental] = []  # 대여 이력을 남기기 위해 반납된 기록까지 보관

    # 내부 도우미: 찾지 못하면 예외
    def _get_book(self, isbn: str) -> Book:  # 도서 찾기와 오류 처리를 한곳에 두기 위해
        if isbn not in self.books:  # 없는 도서를 오류로 알리기 위해
            raise BookNotFoundError(isbn)
        return self.books[isbn]  # 찾은 도서를 돌려주기 위해
    def _get_member(self, name: str) -> Member:  # 회원 찾기와 오류 처리를 한곳에 두기 위해
        if name not in self.members:  # 없는 회원을 오류로 알리기 위해
            raise MemberNotFoundError(name)
        return self.members[name]  # 찾은 회원을 돌려주기 위해
    def _active_rental(self, book: Book, member: Member) -> Rental | None:  # 대여 · 반납 검사에 함께 쓰기 위해
        return next((r for r in self.rentals
                     if r.book is book and r.member is member and r.return_date is None), None)  # 반납 전 기록을 찾기 위해 (없으면 None)

    # 도서
    def add_book(self, title: str, author: str, publication_year: int, isbn: str) -> None:  # 새 도서와 같은 도서를 구분해 등록하기 위해
        if isbn in self.books:  # 같은 책은 새로 만들지 않고 권수만 늘리기 위해
            book = self.books[isbn]  # 기존 도서의 권수를 바꾸기 위해
            book.total_copies += 1  # 보유 권수를 늘리기 위해
            book.available_copies += 1  # 대여 가능 권수도 함께 늘리기 위해
            print(f"'{title}' 도서가 1권 더 추가되었습니다. (보유 {book.total_copies}권)")
            return
        self.books[isbn] = Book(title, author, publication_year, isbn)  # 새 도서를 등록하기 위해
        print(f"'{title}' (저자: {author}, 출판년도: {publication_year}) 도서가 추가되었습니다.")  # 처리 결과를 알리기 위해
    @report_errors  # 없는 도서 · 대여 중 오류를 안내 문구로 바꾸기 위해
    def remove_book(self, isbn: str) -> None:  # 대여 중이 아닌 도서만 지우기 위해
        book = self._get_book(isbn)  # 없는 도서면 예외로 안내하기 위해
        if book.available_copies < book.total_copies:  # 대여 중인 책이 사라지지 않도록
            raise LibraryError(f"'{book.title}' 도서는 대여 중이라 삭제할 수 없습니다.")
        del self.books[isbn]  # 목록에서 지우기 위해
        print(f"'{book.title}' 도서가 삭제되었습니다.")
    def search_books(self, keyword: str) -> list[Book]:  # 제목 · 저자로 도서를 찾기 위해
        found = [b for b in self.books.values() if keyword in b.title or keyword in b.author]  # 제목 · 저자 어느 쪽으로도 찾기 위해
        print(f"'{keyword}' 검색 결과: {len(found)}건")  # 검색 건수를 알리기 위해
        for b in found:  # 대여 가능 여부까지 보이기 위해
            print(f"- {b} [대여 가능 {b.available_copies}/{b.total_copies}권]")
        return found  # 검색 결과를 다른 곳에서도 쓰기 위해

**결과 해설**

```
출력 없음 : 도서 관리 클래스 정의, 다음 셀에서 회원 · 대여 기능을 더해 상속
```

**소속별 최종 요약표**

| 함수·문법 | 소속 | 역할 | 코드에서 사용한 위치 |
| :-- | :-- | :-- | :-- |
| LibraryCatalog | 사용자 정의 클래스 | 도서 추가 · 삭제 · 검색 | `class LibraryCatalog:` |
| values() | dict | 저장된 도서 | `self.books.values()` |
| next() | Python 내장 함수 | 조건에 맞는 첫 기록 (없으면 None) | `next((r for r in ...), None)` |
| 데코레이터 · raise · del | Python 기본 문법 | 오류 안내 씌우기 · 오류 발생 · 삭제 | `@report_errors` · `del self.books[isbn]` |

### 5-3. 도서관 관리 시스템 · 회원 · 대여 클래스

**한글 요약**

`LibraryCatalog`를 상속한 `LibraryManagement`에 목록 출력 · 회원 등록 · 대여 · 반납 추가. 모든 검사를 통과한 뒤에만 재고 · 기록을 바꿈.

**실행 흐름**

1. **입력**: 없음
2. **처리**: 목록 출력 → 중복 없는 회원 등록 → 중복 대여 · 재고 검사 후 대여 → 빌린 책만 반납
3. **저장**: `LibraryManagement` 클래스
4. **출력**: 없음 (정의 셀)

In [25]:
# 회원 · 대여 관리
class LibraryManagement(LibraryCatalog):  # 도서 관리 기능을 그대로 물려받기 위해 상속
    # 목록 출력
    def print_books(self) -> None:  # 도서 목록을 확인하기 위해
        print("도서 목록:")  # 출력 구역을 구분하기 위해
        for book in self.books.values():  # 등록 순서대로 보이기 위해
            print(f"- {book}")

    # 회원
    def add_member(self, name: str) -> None:  # 중복 없이 회원을 등록하기 위해
        if name in self.members:  # 같은 회원이 두 번 등록되지 않도록
            print(f"오류: 회원 '{name}'님은 이미 등록되어 있습니다.")
            return
        self.members[name] = Member(name, len(self.members) + 1)  # 등록 순서를 회원번호로 쓰기 위해
        print(f"회원 '{name}'님이 등록되었습니다.")  # 처리 결과를 알리기 위해
    def print_members(self) -> None:  # 회원별 대여 상태를 확인하기 위해
        print("회원 목록:")  # 출력 구역을 구분하기 위해
        for member in self.members.values():  # 대여 상태와 함께 보이기 위해
            print(f"- {member}")

    # 대여 · 반납
    @report_errors  # 대여 오류를 안내 문구로 바꾸기 위해
    def rent_book(self, isbn: str, member_name: str) -> None:  # 모든 검사를 통과한 대여만 기록하기 위해
        book, member = self._get_book(isbn), self._get_member(member_name)  # 없는 도서 · 회원을 예외로 안내하기 위해
        if self._active_rental(book, member):  # 같은 책을 두 번 빌리지 않도록
            raise LibraryError(f"'{member_name}' 회원님은 이미 '{book.title}' 도서를 대여 중입니다.")
        if book.available_copies == 0:  # 남은 책이 없으면 거부하기 위해
            raise BookNotAvailableError(book.title)
        book.available_copies -= 1  # 대여 가능 권수를 줄이기 위해
        member.rented_books.append(book)  # 회원의 대여 목록에 넣기 위해
        self.rentals.append(Rental(book, member))  # 대여 기록을 남기기 위해
        print(f"'{member_name}' 회원님이 '{book.title}' 도서를 대여하였습니다.")
    @report_errors  # 반납 오류를 안내 문구로 바꾸기 위해
    def return_book(self, isbn: str, member_name: str) -> None:  # 빌린 책만 반납 처리하기 위해
        book, member = self._get_book(isbn), self._get_member(member_name)  # 없는 도서 · 회원을 예외로 안내하기 위해
        rental = self._active_rental(book, member)  # 반납할 대여 기록을 찾기 위해
        if rental is None:  # 빌리지 않은 책 반납을 막기 위해
            raise NotRentedError(member_name, book.title)
        rental.return_date = date.today()  # 모든 검사를 통과한 뒤에만 상태를 바꾸기 위해
        book.available_copies += 1  # 대여 가능 권수를 되돌리기 위해
        member.rented_books.remove(book)  # 회원의 대여 목록에서 빼기 위해
        print(f"'{member_name}' 회원님이 '{book.title}' 도서를 반납하였습니다.")

**결과 해설**

```
출력 없음 : 회원 · 대여 관리 클래스 정의, 다음 셀에서 기본 · 오류 시나리오 실행
```

**소속별 최종 요약표**

| 함수·문법 | 소속 | 역할 | 코드에서 사용한 위치 |
| :-- | :-- | :-- | :-- |
| LibraryManagement | 사용자 정의 클래스 | 도서 관리 + 회원 · 대여 관리 | `class LibraryManagement(LibraryCatalog):` |
| append() · remove() | list | 대여 목록 추가 · 삭제 | `member.rented_books.remove(book)` |
| len() | Python 내장 함수 | 등록 순서 회원번호 | `len(self.members) + 1` |
| 상속 · 데코레이터 · is | Python 기본 문법 | 기능 물려받기 · 오류 안내 · 같은 객체 비교 | `@report_errors` · `r.book is book` |

### 5-3. 도서관 관리 시스템 · 실행

**한글 요약**

문제의 기본 시나리오와 오류 시나리오 6가지 실행. 실패한 시도는 대여 기록 · 재고를 바꾸지 않는지 `assert`로 확인.

**실행 흐름**

1. **입력**: 도서 2권 · 회원 2명, 대여 · 반납 · 검색 · 삭제 시도
2. **처리**: 기본 시나리오 → 오류 시나리오 6가지 → 기록 수 · 재고 검증
3. **저장**: `library`
4. **출력**: 처리 · 오류 안내, 도서 · 회원 목록, 검색 결과, 검증 결과

In [26]:
# 기본 시나리오 (문제의 예시)
library = LibraryManagement()  # 도서관 객체를 만들기 위해
library.add_book("1984", "조지 오웰", 1949, "978-0451524935")  # 예시 도서를 등록하기 위해
library.add_book("앵무새 죽이기", "하퍼 리", 1960, "978-0446310789")  # 예시 도서를 등록하기 위해
library.add_member("홍길동")  # 예시 회원을 등록하기 위해
library.rent_book("978-0451524935", "홍길동")  # 정상 대여를 보기 위해
library.print_books()  # 도서 목록을 확인하기 위해
library.print_members()  # 대여 중인 도서가 표시되는지 보기 위해
library.return_book("978-0451524935", "홍길동")  # 정상 반납을 보기 위해
library.print_members()  # 반납 후 "없음"으로 바뀌는지 보기 위해
print()  # 두 시나리오의 출력을 구분하기 위해

# 오류 시나리오
library.add_member("김철수")  # 재고 없음 상황을 만들기 위해 두 번째 회원 등록
library.rent_book("000-0000000000", "홍길동")  # 없는 도서 안내를 보기 위해
library.rent_book("978-0451524935", "박민수")  # 없는 회원 안내를 보기 위해
library.rent_book("978-0451524935", "홍길동")  # 중복 대여 상황을 만들기 위해 먼저 정상 대여
library.rent_book("978-0451524935", "홍길동")  # 중복 대여 거부를 보기 위해
library.rent_book("978-0451524935", "김철수")  # 1권뿐인 책의 재고 없음 안내를 보기 위해
library.return_book("978-0446310789", "김철수")  # 빌리지 않은 책 반납 안내를 보기 위해
library.search_books("오웰")  # 저자 검색과 대여 가능 권수를 보기 위해
library.remove_book("978-0451524935")  # 대여 중인 책 삭제 거부를 보기 위해

# 검증
assert len(library.rentals) == 2  # 정상 대여 2건만 기록되고 실패한 시도는 빠졌는지 확인하기 위해
assert library.books["978-0451524935"].available_copies == 0  # 삭제가 거부되어 도서가 남았는지 확인하기 위해
print("✅ 통과")  # 검증을 통과했음을 보이기 위해

'1984' (저자: 조지 오웰, 출판년도: 1949) 도서가 추가되었습니다.


'앵무새 죽이기' (저자: 하퍼 리, 출판년도: 1960) 도서가 추가되었습니다.
회원 '홍길동'님이 등록되었습니다.
'홍길동' 회원님이 '1984' 도서를 대여하였습니다.
도서 목록:
- 1984 (저자: 조지 오웰, 출판년도: 1949)
- 앵무새 죽이기 (저자: 하퍼 리, 출판년도: 1960)
회원 목록:
- 홍길동 (대여 중인 도서: 1984)
'홍길동' 회원님이 '1984' 도서를 반납하였습니다.
회원 목록:
- 홍길동 (대여 중인 도서: 없음)

회원 '김철수'님이 등록되었습니다.
오류: ISBN '000-0000000000' 도서를 찾을 수 없습니다.
오류: 회원 '박민수'님을 찾을 수 없습니다.
'홍길동' 회원님이 '1984' 도서를 대여하였습니다.
오류: '홍길동' 회원님은 이미 '1984' 도서를 대여 중입니다.
오류: '1984' 도서는 모두 대여 중입니다.
오류: '김철수' 회원님은 '앵무새 죽이기' 도서를 대여하고 있지 않습니다.
'오웰' 검색 결과: 1건
- 1984 (저자: 조지 오웰, 출판년도: 1949) [대여 가능 0/1권]
오류: '1984' 도서는 대여 중이라 삭제할 수 없습니다.
✅ 통과


**결과 해설**

```
기본 시나리오         : 도서 2권 · 회원 1명 등록 → 대여 → 목록에 '1984' 표시 → 반납 후 "없음"
없는 도서 · 회원      : BookNotFoundError · MemberNotFoundError → "오류: ..." 안내
중복 대여 · 재고 없음 : 이미 빌린 책 거부 · 1권뿐인 책 거부 (BookNotAvailableError)
빌리지 않은 책 반납   : NotRentedError
검색 '오웰'           : 1건, 대여 가능 0/1권
대여 중인 책 삭제     : 거부 → 도서 유지
✅ 통과               : 대여 기록 2건 = 기본 1건 + 오류 시나리오의 정상 대여 1건
```

**소속별 최종 요약표**

| 함수·문법 | 소속 | 역할 | 코드에서 사용한 위치 |
| :-- | :-- | :-- | :-- |
| LibraryManagement | 사용자 정의 클래스 | 도서 · 회원 · 대여 관리 | `library = LibraryManagement()` |
| rent_book() · return_book() | 사용자 정의 메서드 | 대여 · 반납 (오류는 안내) | `library.rent_book("978-0451524935", "홍길동")` |
| len() · print() | Python 내장 함수 | 대여 기록 수 · 출력 구분 | `len(library.rentals)` · `print()` |
| assert | Python 기본 문법 | 실패한 시도가 상태를 바꾸지 않았는지 검사 | `assert len(library.rentals) == 2` |

---
## 6. 마무리

### 핵심 결과

- **19문제 모두 검증 통과**: 다른 방법으로 구한 답(반복문 합 · groupby · 하나씩 판별한 소수 합)이나 기대값과 `assert`로 비교
- **입력 오류에 멈추지 않음**: 숫자 아님 · 0으로 나누기 · 잘못된 날짜 · 잔액 초과 · 없는 도서 · 회원을 모두 안내 문구로 처리
- **더 나은 풀이 선택**: 총합은 가우스 공식(O(1)), 소수 합은 에라토스테네스의 체, 시간 관리는 그리디 + 조합으로 남는 시간 최소화
- **상태를 지키는 클래스**: 잔액은 메서드로만 변경, 실패한 거래 · 대여는 상태를 바꾸지 않음, 클래스 변수 중복 저장 방지
- **한 번에 재실행**: `fake_inputs`로 모든 `input()`을 자동 입력 → Restart & Run All 한 번으로 전체 결과 재현

### 최종 요약표

| 파트 | 문제 | 핵심 기법 |
| :-- | :-- | :-- |
| **0. 준비** | 입력 자동화 | `contextmanager` · `try · finally`로 `input()` 교체 · 복구 |
| **1. 파이썬 기초** | 3의 배수 · 모음 · 비교 · 길이 · 총합 · 짝수 합 · 나누기 | `range` 간격, `set` 포함 검사, 가우스 공식, `try · except` 3종류 |
| **2. 파이썬 응용** | 압축 · 덧셈 · 타이머 | 이웃 글자 비교 + `groupby` 검증, 재사용 입력 함수 `read_number` |
| **3. 클래스 기초** | 시간 추적 · 주소록 | `total_seconds()`, `__str__` 출력 형식 |
| **4. 클래스 응용** | 투표 · 계좌 · 직원 · 예약 | 거부 규칙, `@staticmethod` · `@classmethod`, 날짜 표기 통일 |
| **5. 심화** | 시간 관리 · 소수 합 · 도서관 | 그리디 + 조합, 에라토스테네스의 체, 사용자 정의 예외 + 데코레이터 |